# 2.0 · Selección y comprensión del dataset

**Proyecto JobPath · Minería de Datos · UNIMINUTO Ibagué · 2026-2**

Este cuaderno abre la fase de **Preparación de los datos** de CRISP-DM. No
explora todavía: **selecciona, entiende y valida**. El análisis exploratorio
completo vive en `5.0_EDA.ipynb`.

> Reemplaza a los antiguos `2.0_EDA_y_seleccion` y `2.1_EDA_informe`. Aquí solo
> queda lo que corresponde al paso 2: qué datos hay, qué significa cada columna
> y qué reglas se aplican antes de transformar.

## Alcance

| Sí hace | No hace |
|---|---|
| Inventario de fuentes y columnas | Gráficos de EDA (van en `5.0`) |
| Construcción de `Period_Time` | Escalado y codificación (van en `4.0`) |
| Banderas ESCO y sus validaciones | PCA (va en `4.3`) |
| Medición de vacíos, duplicados y outliers | Borrar filas o imputar |

**Regla del proyecto:** los CSV son de solo lectura. Ninguna celda los
sobrescribe. Si un dato se corrige, se corrige en memoria y se documenta.

In [1]:
from pathlib import Path
import sys


def _carpeta_refactor() -> Path:
    """Ubica notebooks/refactor sin depender del directorio de ejecucion."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "lib" / "config.py").exists():
            return p
        if (p / "notebooks" / "refactor" / "lib" / "config.py").exists():
            return p / "notebooks" / "refactor"
    raise RuntimeError("No se encontro la carpeta notebooks/refactor")


RAIZ_CUADERNO = _carpeta_refactor()
if str(RAIZ_CUADERNO) not in sys.path:
    sys.path.insert(0, str(RAIZ_CUADERNO))

import pandas as pd
from IPython.display import display

from lib import calidad, carga, competencias, config, periodo

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
print("Paquete cargado desde:", RAIZ_CUADERNO)

Paquete cargado desde: C:\Users\ACER\Desktop\MineriaProtect\MineriaProtect\notebooks\refactor


## 1. Fuentes disponibles

Hay dos versiones del mismo dataset integrado. La unica diferencia es el
tratamiento de la duracion: una conserva la cola larga y la otra la excluye.
Se cuentan las filas **sin cargar el archivo completo**, para no gastar memoria
en un dato que solo se quiere dimensionar.

In [2]:
filas = {nombre: carga.contar_filas(f) for nombre, f in config.FUENTES.items()}

resumen_fuentes = pd.DataFrame(
    [
        {
            "fuente": nombre,
            "archivo": fuente.ruta.name,
            "filas": filas[nombre],
            "criterio": fuente.criterio,
        }
        for nombre, fuente in config.FUENTES.items()
    ]
)
display(resumen_fuentes)

n_con = filas["con_outliers"]
n_sin = filas["sin_outliers"]
print(f"Diferencia: {n_con - n_sin:,} filas ({100 * (n_con - n_sin) / n_con:.2f} % del total).")

,fuente,archivo,filas,criterio
0,con_outliers,empleos_limpio_con_competencias.csv,376567,Conserva todas las filas; la cola larga de dur...
1,sin_outliers,empleos_con_competencias_limpio_sin_outliers.csv,341090,Excluye las filas con duracion fuera del limit...


Diferencia: 35,477 filas (9.42 % del total).


La diferencia es de **35.477 filas (9,42 %)**. Esa es exactamente la cola
larga de duracion que el archivo `sin_outliers` dejo fuera al aplicar el limite
de Tukey sobre la duracion trimestral.

**Decision postergada a 3.0:** aqui solo se dimensiona la diferencia. En `3.0`
se revisa que son esas 35.477 filas (duraciones largas reales, no errores) y se
decide con evidencia. El criterio del curso es no borrar outliers, solo
ocultarlos.

## 2. Diccionario de columnas

El rol no se deduce del `dtype`. Una columna de texto puede ser un
identificador, una categoria o una lista; tratarlas igual seria el primer
error. El diccionario vive en `lib/columnas.py` y se cita aqui.

In [3]:
df = carga.leer(config.SIN_OUTLIERS)
print("Forma:", df.shape)

inventario = carga.inventario(df)
display(inventario)

Forma: (341090, 23)


,columna,rol,dtype,vacios,distintos,entra_a_x,descripcion
0,resume_id,identificador,str,0,69182,no,Persona duena de la hoja de vida; no es un pat...
1,start_date,temporal,str,0,204,no,Trimestre de inicio de la experiencia (Qn AAAA).
2,end_date,temporal,str,0,204,no,Trimestre de fin de la experiencia (Qn AAAA).
3,university_level,ordinal,str,37868,4,si,Maximo nivel academico: No reportado < ... < PhD.
4,matched_code,identificador,str,0,2844,no,Codigo con el que se intento el emparejamiento...
5,emparejado,nominal,str,0,3,si,"Resultado del emparejamiento: ok, unknown o re..."
6,occupation_code,nominal_alta_cardinalidad,str,26186,2826,condicional,Codigo ESCO del oficio (2.855 valores).
7,occupation_label,texto_redundante,str,26186,2826,no,Nombre del oficio; 1:1 con occupation_code.
8,isco_group,nominal,str,23886,424,si,Gran grupo ISCO de un digito (9 valores).
9,isco_group_label,texto_redundante,str,23886,424,no,Nombre del grupo ISCO; 1:1 con isco_group.


Lectura del inventario:

- **3 identificadores** (`resume_id`, `matched_code`, `occupation_code`): no
  describen un patron, solo nombran.
- **2 columnas temporales** (`start_date`, `end_date`): se convierten en
  `Period_Time` y en duracion, nunca entran crudas.
- **2 textos redundantes** (`occupation_label`, `isco_group_label`): son 1:1 con
  su codigo; aportan lectura, no informacion nueva.
- **1 constante** (`isco_level` = 4): no puede separar nada.
- **4 conteos ESCO** y **3 binarias** de calidad del emparejamiento.

Las listas (`skills`, `knowledge`, `sin_tipo`) no entran a `X`: son texto con
miles de valores distintos. Su informacion ya esta resumida en los contadores.

## 3. `Period_Time`: de dos columnas a una

`start_date` y `end_date` vienen como trimestres (`Q1 2011`). Unirlas en una
sola etiqueta (`Q1 2011 - Q1 2015`) sirve para leer y para ordenar, pero **no**
las reemplaza para calcular: la duracion necesita los ordinales por separado.

`end_date` puede valer `Present` cuando la experiencia sigue vigente. Ese caso
**no se imputa**: la duracion observada queda vacia y se calcula aparte una
duracion al corte.

In [4]:
periodo.validar(df)
df["period_time"] = periodo.crear_period_time(df)

display(df[["resume_id", "start_date", "end_date", "period_time"]].head(5))

,resume_id,start_date,end_date,period_time
0,10,Q1 2011,Q1 2015,Q1 2011 - Q1 2015
1,10,Q1 2014,Q1 2015,Q1 2014 - Q1 2015
2,10,Q1 2015,Q3 2017,Q1 2015 - Q3 2017
3,10,Q1 2016,Present,Q1 2016 - Present
4,100013,Q1 2007,Q1 2008,Q1 2007 - Q1 2008


In [5]:
duracion = periodo.resumen_duracion(df)
display(pd.DataFrame([duracion]).T.rename(columns={0: "valor"}))

corte = periodo.trimestre_de_corte(df)
anio, trimestre = divmod(corte, 4)
print(f"Trimestre de corte: Q{trimestre} {anio}")

,valor
n_con_fin,322134.0
n_vigentes,18956.0
min,2.0
q1,2.0
mediana,5.0
q3,9.0
max,24.0


Trimestre de corte: Q1 2024


Hay **18.956 experiencias vigentes** (`Present`). Para ellas:

- `duracion_observada` queda vacia, porque inventar un fin seria afirmar algo
  que el dato no dice.
- `duracion_a_corte` usa el ultimo trimestre observado como referencia.

**Riesgo de fuga:** `Period_Time` no puede ser predictor. Contiene la fecha de
fin, que en un problema de abandono es parte de la respuesta. Se usa para el
EDA y para derivar duracion, nada mas.

## 4. Banderas ESCO y el riesgo de falso positivo

La instruccion original pedia marcar como "Sin Clasificar" toda lista vacia.
Aplicada al pie de la letra, esa regla miente: hay dos causas distintas de
vacio y solo una significa "no clasificado".

| Hecho | Filas | Que significa |
|---|---|---|
| No hay oficio emparejado | 26.186 | No se sabe nada del oficio |
| Hay oficio y no exige conocimientos | 32.967 | Si se sabe: es un cargo de puro hacer |
| Hay oficio y no exige acciones | 2 | Caso excepcional, mismo significado |

Por eso se construyen **dos banderas de tres valores**, no una binaria. Ambas
comparten `sin_clasificar` (no hay oficio) y se diferencian en el resto.

In [6]:
df = competencias.agregar_banderas(df)
competencias.validar(df)
print("Validacion superada: las banderas coinciden con saber_skills; es_unknown_ocupacion es un subconjunto (no incluye rescatado).")

display(competencias.resumen_banderas(df))

Validacion superada: las banderas coinciden con saber_skills; es_unknown_ocupacion es un subconjunto (no incluye rescatado).


,bandera,valor,filas,porcentaje
0,estado_competencias,con_competencias,314902,92.32
1,estado_competencias,sin_clasificar,26186,7.68
2,estado_competencias,sin_competencias,2,0.00
3,estado_conocimientos,con_conocimientos,281937,82.66
4,estado_conocimientos,sin_conocimientos,32967,9.67
5,estado_conocimientos,sin_clasificar,26186,7.68


El cruce demuestra que no son la misma bandera: las 26.186 caen en la misma
celda, pero las 32.967 aparecen solo del lado de conocimientos.

In [7]:
display(competencias.cruce_banderas(df))

estado_conocimientos,con_conocimientos,sin_clasificar,sin_conocimientos,total
estado_competencias,,,,
con_competencias,281935,0,32967,314902
sin_clasificar,0,26186,0,26186
sin_competencias,2,0,0,2
total,281937,26186,32967,341090


### Cuanto pesa la regla ingenua

La regla "lista vacia = sin clasificar" marcaria 59.155 filas. La regla correcta
marca 26.186. La diferencia son 32.969 oficios validos mal etiquetados.

In [8]:
auditoria = competencias.auditar_falso_positivo(df)

tabla_riesgo = pd.DataFrame(
    [
        ("Sin clasificar (regla correcta: saber_skills)", auditoria["verdaderos_sin_clasificar"]),
        ("Sin clasificar (regla ingenua: lista vacia)", auditoria["ingenuos_sin_clasificar"]),
        ("Falsos positivos de la regla ingenua", auditoria["falsos_positivos"]),
        ("Oficios sin conocimientos (hecho de negocio)", auditoria["sin_conocimientos_reales"]),
        ("Oficios sin competencias (hecho de negocio)", auditoria["sin_competencias_reales"]),
    ],
    columns=["conteo", "filas"],
)
display(tabla_riesgo)

,conteo,filas
0,Sin clasificar (regla correcta: saber_skills),26186
1,Sin clasificar (regla ingenua: lista vacia),59155
2,Falsos positivos de la regla ingenua,32969
3,Oficios sin conocimientos (hecho de negocio),32967
4,Oficios sin competencias (hecho de negocio),2


### Riesgos y mitigaciones

| # | Riesgo | Que lo provoca | Mitigacion implementada |
|---|---|---|---|
| 1 | Falso positivo "sin clasificar" | Derivar la bandera del vacio de la lista | Se deriva de `saber_skills` (`competencias.sin_oficio`); `validar()` lo impone |
| 2 | Bandera binaria ambigua | Un `False` no distingue "no exige" de "no aplica" | Dos banderas de tres valores |
| 3 | Doble conteo en el informe | Sumar 26.186 + 32.967 | `cruce_banderas()` y secciones separadas en el informe |
| 4 | Fuga temporal | Usar `Period_Time` como predictor | Solo EDA y calculo de duracion |
| 5 | Borrado silencioso de outliers | Confundir "ocultar" con "eliminar" | `calidad.ocultar()` recorta solo la serie graficada |
| 6 | Nombres inexistentes en la instruccion | `skill_knowledge` / `skill_competence` no existen | `lib/columnas.py` declara los nombres reales |

**Prueba de que la mitigacion funciona:** el `assert` de `validar()` se cae si
las banderas no coinciden con `saber_skills`. No es un comentario: es
una condicion ejecutable.

### Los contadores son el largo de las listas

Esta comprobacion cierra el circulo: si `n_skills_competence` no fuera el
numero de elementos de `skills`, las banderas y los conteos podrian contar
historias distintas.

In [9]:
display(competencias.validar_contadores(df))

,comprobacion,coinciden,total,resultado
0,skills vs n_skills_competence,314904,314904,OK
1,knowledge vs n_skills_knowledge,314904,314904,OK
2,sin_tipo vs n_skills_sin_tipo,314904,314904,OK


## 5. Calidad: medir antes de decidir

Se miden vacios, duplicados y outliers. **No se imputa y no se elimina**: eso
corresponde a `3.0`, con su justificacion por columna.

In [10]:
display(calidad.resumen_vacios(df))
display(pd.DataFrame([calidad.resumen_duplicados(df)]).T.rename(columns={0: "valor"}))
display(calidad.resumen_outliers(df, ["n_skills_essential", "veces_ese_oficio"]))

,vacios,porcentaje
sin_tipo,338144,99.14
knowledge,59153,17.34
university_level,37868,11.10
skills,26188,7.68
occupation_label,26186,7.68
occupation_code,26186,7.68
n_skills_sin_tipo,26186,7.68
n_skills_knowledge,26186,7.68
n_skills_competence,26186,7.68
n_skills_essential,26186,7.68


,valor
filas,341090
duplicados_exactos,0
resume_id_distintos,69182
experiencias_por_persona_max,20


,columna,q1,q3,iqr,limite_inferior,limite_superior,outliers,porcentaje
0,n_skills_essential,15.0,28.0,13.0,-4.5,47.5,15205,4.46
1,veces_ese_oficio,1.0,2.0,1.0,-0.5,3.5,20219,5.93


Los vacios de `skills`/`knowledge` no son un problema de imputacion: son
el reflejo de las 26.186 filas sin oficio. Imputar ahi seria inventar
competencias para un cargo que no se conoce.

Los outliers de `n_skills_essential` y `veces_ese_oficio` se conservan. En el
EDA se ocultan solo en el grafico, con `calidad.ocultar()`, para que la cola no
aplaste la lectura.

## 6. Seleccion: que entra a `X` y que no

| Columna | Rol | Entra a `X` | Decision |
|---|---|---|---|
| `resume_id` | identificador | No | Identifica personas, no patrones |
| `start_date`, `end_date` | temporal | No | Se derivan `Period_Time` y duracion |
| `occupation_label`, `isco_group_label` | texto 1:1 | No | Redundantes con su codigo |
| `isco_level` | constante | No | Vale 4 en todas las filas |
| `occupation_code` | nominal 2.855 | Condicional | Se agrupa a `isco_group` (9 valores) |
| `skills`, `knowledge`, `sin_tipo` | lista de texto | No | Se resumen en contadores |
| `n_skills_competence`, `_knowledge`, `_sin_tipo` | conteo | Condicional | Redundantes con `n_skills_essential`; se prefiere el ratio |
| `period_time`, `estado_*` | derivadas | No | Son de lectura y control, no predictoras |

La matriz final se construye en `4.0`. Aqui solo se declara la intencion, para
que la transformacion no improvise.

## 7. Decisiones de este cuaderno

1. **Fuentes:** se dimensionan ambas; la base se elige en `3.0` con evidencia
   sobre las 35.477 filas de la cola.
   La diferencia es de 35.477 filas (9,42 %) y se documenta.
2. **`Period_Time`** se construye por concatenacion; no reemplaza a las fechas
   originales, que siguen disponibles para calcular duracion.
3. **Vigentes:** 18.956 filas con `Present`; su duracion observada queda vacia.
4. **Banderas:** dos, de tres valores, derivadas de `saber_skills`.
5. **Vacios de listas:** no se imputan; son ausencia estructural.
6. **Outliers:** se miden y se conservan; se ocultan solo al graficar.
7. **CSV:** no se modifican.

## 8. Reproducibilidad

- Rutas resueltas desde `lib/config.py`, sin rutas absolutas.
- Lectura con `dtype=str` y `na_values=[""]`: los codigos conservan ceros a la
  izquierda y los vacios no se vuelven cero.
- Validaciones ejecutables (`periodo.validar`, `competencias.validar`).
- Para ejecutar todo:

```bash
jupyter nbconvert --to notebook --execute --inplace 2.0_seleccion_dataset.ipynb
```